# Week 3 · Class 2 — Filtering & Sorting Data
**AI Training Program · Phase 1: Python Foundations**

---

## What you'll learn
- `WHERE` clause with comparison and logical operators
- Pattern matching with `LIKE` and `GLOB`
- Range filtering: `BETWEEN`, `IN`, `NOT IN`
- Handling `NULL` values correctly
- `ORDER BY` (single and multi-column) and `LIMIT` / `OFFSET`
- `DISTINCT` to remove duplicate values
- `CASE WHEN` for conditional logic inside SQL

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("school.db")
cursor = conn.cursor()

# Re-build data in case school.db doesn't exist yet
cursor.executescript("""
    DROP TABLE IF EXISTS students;
    DROP TABLE IF EXISTS courses;
    DROP TABLE IF EXISTS enrollments;

    CREATE TABLE students (
        student_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        name         TEXT    NOT NULL,
        email        TEXT    UNIQUE NOT NULL,
        age          INTEGER,
        city         TEXT    DEFAULT 'Mumbai',
        gpa          REAL,
        enrolled_on  TEXT    DEFAULT (date('now'))
    );

    INSERT INTO students (name, email, age, city, gpa) VALUES
        ('Aarav Shah',    'aarav@example.com',  24, 'Mumbai',    3.8),
        ('Priya Nair',    'priya@example.com',  22, 'Pune',      3.5),
        ('Rohan Mehta',   'rohan@example.com',  26, 'Delhi',     2.9),
        ('Sana Khan',     'sana@example.com',   23, 'Mumbai',    3.7),
        ('Dev Sharma',    'dev@example.com',    28, 'Bangalore', 3.1),
        ('Anita Bose',    'anita@example.com',  21, 'Kolkata',   3.9),
        ('Kiran Rao',     'kiran@example.com',  25, 'Hyderabad', NULL),
        ('Meera Pillai',  'meera@example.com',  20, 'Mumbai',    3.6),
        ('Arjun Gupta',   'arjun@example.com',  27, 'Delhi',     2.7),
        ('Sneha Iyer',    'sneha@example.com',  22, 'Chennai',   3.4);

    CREATE TABLE courses (
        course_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        title      TEXT NOT NULL,
        instructor TEXT NOT NULL,
        credits    INTEGER DEFAULT 3,
        dept       TEXT
    );

    INSERT INTO courses (title, instructor, credits, dept) VALUES
        ('Python Foundations',  'Dr. Mehta',  4, 'CS'),
        ('Machine Learning',    'Dr. Rao',    4, 'AI'),
        ('SQL & Databases',     'Dr. Khan',   3, 'CS'),
        ('Deep Learning',       'Dr. Sharma', 4, 'AI'),
        ('MLOps & Deployment',  'Dr. Nair',   3, 'AI'),
        ('Statistics for AI',   'Dr. Bose',   3, 'Math');
""")
conn.commit()
print("Database ready ✓")
pd.read_sql_query("SELECT * FROM students", conn)

## 1. The `WHERE` Clause

WHERE filters **which rows** are returned. Think of it as `if` for rows.

### Comparison operators
| Operator | Meaning |
|---|---|
| `=` | Equal to |
| `!=` or `<>` | Not equal to |
| `>`, `>=` | Greater than (or equal) |
| `<`, `<=` | Less than (or equal) |

In [ ]:
# Students older than 24
pd.read_sql_query("""
    SELECT name, age, city
    FROM   students
    WHERE  age > 24
""", conn)

In [ ]:
# Students from a specific city
pd.read_sql_query("""
    SELECT name, city, gpa
    FROM   students
    WHERE  city = 'Mumbai'
""", conn)

### Logical operators: `AND`, `OR`, `NOT`

In [ ]:
# Mumbai students who are also younger than 24
pd.read_sql_query("""
    SELECT name, age, city, gpa
    FROM   students
    WHERE  city = 'Mumbai'
      AND  age < 24
""", conn)

In [ ]:
# Students from Mumbai OR with GPA above 3.7
pd.read_sql_query("""
    SELECT name, city, gpa
    FROM   students
    WHERE  city = 'Mumbai'
       OR  gpa > 3.7
""", conn)

In [ ]:
# NOT — students NOT from Mumbai
pd.read_sql_query("""
    SELECT name, city
    FROM   students
    WHERE  NOT city = 'Mumbai'
""", conn)

## 2. `BETWEEN` — Range Filtering

In [ ]:
# Students aged 22 to 25 inclusive
pd.read_sql_query("""
    SELECT name, age
    FROM   students
    WHERE  age BETWEEN 22 AND 25
    ORDER BY age
""", conn)

In [ ]:
# GPA between 3.4 and 3.8
pd.read_sql_query("""
    SELECT name, gpa
    FROM   students
    WHERE  gpa BETWEEN 3.4 AND 3.8
    ORDER BY gpa DESC
""", conn)

## 3. `IN` and `NOT IN` — Membership Test

In [ ]:
# Students from specific cities (more readable than multiple OR)
pd.read_sql_query("""
    SELECT name, city
    FROM   students
    WHERE  city IN ('Mumbai', 'Delhi', 'Bangalore')
    ORDER BY city
""", conn)

In [ ]:
# Students NOT from those cities
pd.read_sql_query("""
    SELECT name, city
    FROM   students
    WHERE  city NOT IN ('Mumbai', 'Delhi', 'Bangalore')
""", conn)

## 4. `LIKE` — Pattern Matching

| Wildcard | Matches |
|---|---|
| `%` | Zero or more of any character |
| `_` | Exactly one character |

In [ ]:
# Names starting with 'A'
pd.read_sql_query("""
    SELECT name FROM students
    WHERE  name LIKE 'A%'
""", conn)

In [ ]:
# Names ending with 'a'
pd.read_sql_query("""
    SELECT name FROM students
    WHERE  name LIKE '%a'
""", conn)

In [ ]:
# Emails containing 'nair' anywhere
pd.read_sql_query("""
    SELECT name, email FROM students
    WHERE  email LIKE '%nair%'
""", conn)

In [ ]:
# Names with exactly 5 characters before 'a' (e.g. 'Priya')
# _ = one character
pd.read_sql_query("""
    SELECT name FROM students
    WHERE  name LIKE '____a%'   -- 4 underscores = 4 exact chars
""", conn)

## 5. Handling `NULL` Values

> **Critical**: `NULL` means *unknown* — you cannot use `= NULL`. You must use `IS NULL` or `IS NOT NULL`.

In [ ]:
# Wrong way — this returns 0 rows even if NULLs exist!
wrong = pd.read_sql_query("SELECT name, gpa FROM students WHERE gpa = NULL", conn)
print("Wrong (= NULL):", len(wrong), "rows")

# Correct way
correct = pd.read_sql_query("SELECT name, gpa FROM students WHERE gpa IS NULL", conn)
print("Correct (IS NULL):")
print(correct)

In [ ]:
# COALESCE — replace NULL with a fallback value
pd.read_sql_query("""
    SELECT
        name,
        gpa,
        COALESCE(gpa, 0.0)     AS gpa_safe,
        COALESCE(gpa, 0.0) * 25 AS score_out_of_100
    FROM students
""", conn)

## 6. `ORDER BY` and `LIMIT` / `OFFSET`

In [ ]:
# Top 3 students by GPA
pd.read_sql_query("""
    SELECT name, gpa
    FROM   students
    WHERE  gpa IS NOT NULL
    ORDER BY gpa DESC
    LIMIT  3
""", conn)

In [ ]:
# Multi-column sort: city ASC, then age DESC within city
pd.read_sql_query("""
    SELECT name, city, age
    FROM   students
    ORDER BY city ASC, age DESC
""", conn)

In [ ]:
# Pagination with OFFSET — page 2 (rows 4-6) of results sorted by name
page_size = 3
page_num  = 2  # 1-based

df_page = pd.read_sql_query(f"""
    SELECT name, age, city
    FROM   students
    ORDER BY name
    LIMIT  {page_size}
    OFFSET {(page_num - 1) * page_size}
""", conn)
print(f"Page {page_num}:")
df_page

## 7. `DISTINCT` — Unique Values

In [ ]:
# All unique cities in the database
pd.read_sql_query("""
    SELECT DISTINCT city
    FROM   students
    ORDER BY city
""", conn)

## 8. `CASE WHEN` — Conditional Logic in SQL

In [ ]:
# Assign letter grades based on GPA
pd.read_sql_query("""
    SELECT
        name,
        gpa,
        CASE
            WHEN gpa >= 3.7 THEN 'A'
            WHEN gpa >= 3.3 THEN 'B'
            WHEN gpa >= 3.0 THEN 'C'
            WHEN gpa IS NOT NULL THEN 'F'
            ELSE 'N/A'
        END AS grade
    FROM students
    ORDER BY gpa DESC
""", conn)

In [ ]:
# Categorise students by age group
pd.read_sql_query("""
    SELECT
        name, age,
        CASE
            WHEN age < 22 THEN 'Junior'
            WHEN age < 26 THEN 'Mid'
            ELSE               'Senior'
        END AS cohort
    FROM students
    ORDER BY age
""", conn)

---

## ✏️ Exercises

**Exercise 1**: Find all students with a GPA strictly above 3.5, ordered highest to lowest.

In [ ]:
# Your code here


**Exercise 2**: Find students whose name contains the letter 'n' (case-insensitive) using `LIKE`.

In [ ]:
# Your code here


**Exercise 3**: List students aged 22–25 NOT from Mumbai or Delhi, sorted by city then name.

In [ ]:
# Your code here


**Exercise 4**: Add a `scholarship` column using `CASE WHEN`: 'Full' if GPA ≥ 3.8, 'Partial' if GPA ≥ 3.5, 'None' otherwise (handle NULLs).

In [ ]:
# Your code here


---

## Summary

| Clause / Operator | Purpose |
|---|---|
| `WHERE col = val` | Equality filter |
| `AND`, `OR`, `NOT` | Combine conditions |
| `BETWEEN a AND b` | Inclusive range |
| `IN (...)` | Match any value in list |
| `LIKE '%pat%'` | Substring / pattern matching |
| `IS NULL` / `IS NOT NULL` | NULL-safe checks |
| `COALESCE(col, default)` | Replace NULL with fallback |
| `ORDER BY col DESC` | Sort results |
| `LIMIT n OFFSET k` | Pagination |
| `DISTINCT` | Remove duplicate rows |
| `CASE WHEN` | Conditional column values |

**Next class**: `GROUP BY`, aggregate functions (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`), `HAVING`, and `JOIN`.

In [ ]:
conn.close()